# Calibration of Language Models

When a model says 0.9, is it right nine times in ten? For a next-token distribution over thousands of words the
question splits in two: a **canonical** form (the whole distribution is right among the contexts that receive it) that
no corpus can test, and a **top-label** form (the top token's probability is right) that any corpus can. This
notebook makes four points:
- every log loss splits into what the text cannot tell us, what the model's context discards, and what a
  recalibration could recover (Thm 1);
- the binned estimator of calibration error is biased in **both** directions, and a calibrated twin measures its
  noise floor (Thm 2);
- temperature scaling is a convex one-parameter fit that matches one moment, and maximum likelihood leaves it at
  exactly τ = 1 on the training data only when the family can scale its own logits (Prop 2, Thm 3);
- calibration is what makes a nucleus a prediction set, and on real filings it fails (Prop 3, finance case study).

Every number below comes from `language_model_calibration.py`, which owns them; this notebook only calls it.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "language_model_calibration.py").exists()
                       else here / "notebooks" / "language-model-calibration"))
import language_model_calibration as lc

S = lc.summary()
for e in lc.ENGINES:
    print(e, "V =", lc.rows(e, "edgar").Z.shape[1], {t: S[f"{e}_{t}_n"] for t in lc.TEST_SETS}, "tokens")

## 1. What calibration means for a 3,427-way forecast

A context $X$ (the trigram's last two words, or the decoder's prefix), its next token $Y$, the forecast
$F = f(X) \in \Delta^{\mathcal V}$, the truth $P = P(Y \mid X)$, and the **recalibration** $C = P(Y \mid F) = \mathbb E[P \mid F]$.

**Definition 1.** $f$ is *canonically calibrated* if $C = F$; with $\mathrm{conf} = \max_v F_v$ and $\hat a$ the
(lowest-id) argmax, $f$ is *top-label calibrated* if $P(Y = \hat a \mid \mathrm{conf}) = \mathrm{conf}$.

**Proposition 1.** (a) Canonical implies top-label. (b) Not conversely: truth $(0.5, 0.2, 0.3)$, forecast
$(0.5, 0.3, 0.2)$. (c) If $f$ gives every context its own row, then $C = P$: canonical calibration *is* being the true
conditional, and the reliability term of Theorem 1 is the whole excess loss $\mathbb E\,\mathrm{KL}(P \,\|\, F)$.

In [ ]:
lc.test_canonical_implies_top_label()
lc.test_distinct_rows_make_calibration_truth()
cx = lc.counterexample_world()
print("counterexample: top-label error", lc.top_label_ece(cx), " recalibration", lc.recalibrated(cx)[0], "forecast", cx.F[0])

## 2. What miscalibration costs

**Theorem 1.** With every expectation over $X$ and $Y$,
$$\mathbb E[-\log F_Y] = \underbrace{\mathbb E\,H(P)}_{\text{irreducible}} + \underbrace{\mathbb E\,\mathrm{KL}(P \,\|\, C)}_{\text{refinement gap}} + \underbrace{\mathbb E\,\mathrm{KL}(C \,\|\, F)}_{\text{reliability}},
\qquad
\mathbb E\|F - e_Y\|^2 = \mathbb E(1 - \|P\|^2) + \mathbb E\|C - P\|^2 + \mathbb E\|F - C\|^2 .$$
The Brier identity is Pythagoras twice: conditional on $F$ the outcome's mean is $C$, conditional on $X$ it is $P$, so
both cross terms vanish. The log identity is the same picture with KL in place of squared distance.

**Corollary 1.** No map $g$ of the forecast recovers more than the reliability term:
$\mathrm{loss}(g \circ F) \ge \mathrm{loss}(F) - \mathbb E\,\mathrm{KL}(C \,\|\, F)$. On a calibrated forecaster no
temperature helps.

In [ ]:
lc.test_log_and_brier_decompositions()
lc.test_any_recalibration_is_bounded_by_reliability()
w = lc.random_world(np.random.default_rng([lc.SEED, 1990]))
for name, d in (("log", lc.log_decomposition(w)), ("Brier", lc.brier_decomposition(w))):
    print(f"{name:5}  loss {d['loss']:.6f} = {d['irreducible']:.6f} + {d['refinement_gap']:.6f} + {d['reliability']:.6f}"
          f"   recalibrated {d['recalibrated_loss']:.6f}")

## 3. Measuring it honestly

Let $\rho(c) = P(Y = \hat a \mid \mathrm{conf} = c)$. For equal-width bins, the population binned error is
$\mathrm{ECE}_{\mathfrak B} = \sum_b P(b)\,\lvert \mathbb E[\rho(\mathrm{conf}) - \mathrm{conf} \mid b] \rvert$.

**Theorem 2.** (a) *Binning hides:* refining the bins never lowers $\mathrm{ECE}_{\mathfrak B}$, and none exceeds the
true error (Jensen across a bin). (b) *Sampling inflates:* if correctness indicators are independent given the
confidences, the plug-in estimate's expectation is at least the sample's binned error (Jensen within a bin), so a
perfectly calibrated model still shows an error of order $\sqrt{B/n}$. (c) The squared error has an exact correction:
$$\mathbb E\Big[(\bar a_b - \bar c_b)^2 - \tfrac{\bar a_b(1 - \bar a_b)}{n_b - 1}\Big] = (\bar\rho_b - \bar c_b)^2 - \tfrac{\sum_{i \in b}(\rho_i - \bar\rho_b)^2}{n_b(n_b - 1)} .$$

The **calibrated twin** keeps a model's confidences and redraws correctness as $\mathrm{Bernoulli}(\mathrm{conf})$: it is
calibrated by construction, so its ECE is the estimator's noise floor.

In [ ]:
lc.test_binning_hides()
lc.test_sampling_inflates_and_the_square_debiases()
lc.test_calibrated_twin_floor()
lc.test_kneser_ney_reliability_shape()
rng = np.random.default_rng([lc.SEED, 1991])
e = lc.bin_expectations(rng.uniform(0.2, 0.8, 12), rng.uniform(0, 1, 12))
print({k: round(v, 6) for k, v in e.items()})
s = lc.stats_at("trigram", "continuation", 1.0)
tw = lc.twin_ece(s.conf, lc.LAB_BINS, lc.TWIN_R, np.random.default_rng([lc.SEED, lc.TWIN_STREAM, 0, 0]))
for j, B in enumerate(lc.LAB_BINS):
    print(f"B = {B:3d}   trigram ECE {lc.ece_hat(s.conf, s.correct, B):.4f}   twin {tw[:, j].mean():.4f} ± {tw[:, j].std(ddof=1):.4f}")

## 4. One dial: temperature scaling

**Proposition 2.** With $\beta = 1/\tau$ and $L(\beta) = \frac1N \sum[\log \sum_v e^{\beta z_v} - \beta z_y]$:
(a) $L'(\beta) = \frac1N\sum(\mathbb E_{p_\beta} z - z_y)$ and $L''(\beta) = \frac1N\sum \mathrm{Var}_{p_\beta}(z) \ge 0$,
so $L$ is convex; (b) its minimizer is unique and matches one moment, $\sum \mathbb E_{p_\beta} z = \sum z_y$;
(c) every row keeps its argmax, but rows can swap their confidences: $(0.5, 0.5, 0)$ and $(0.6, 0.2, 0.2)$ have
confidences $0.5 \lt 0.6$, and at $\tau = 2$ they have $0.5 \gt 0.464$.

In [ ]:
lc.test_temperature_fit_is_convex_and_matches_moments()
lc.test_temperature_keeps_argmax_not_ranking()
print(f"{'':10}" + "".join(f"{t:>14}" for t in lc.TEST_SETS))
for e in lc.ENGINES:
    print(f"{e:10}" + "".join(f"{S[f'{e}_{t}_tau_star']:>14.6f}" for t in lc.TEST_SETS))
print("Kendall tau_b of trigram confidences, tau = 1 vs 2:", round(S["trigram_continuation_kendall_tau2"], 6))

## 5. Why τ = 1 at all

**Theorem 3.** If a family can scale its own logits ($z_{s_\beta(\theta)} = \beta z_\theta$, with a penalty that scales as
$\Omega(s_\beta\theta) = \beta^2\Omega(\theta)$), then at any stationary point $\hat\theta$ of $L_{\text{train}} + \lambda\Omega$,
$$\frac{d}{d\beta} L_{\text{train}}(\beta z_{\hat\theta})\Big|_{\beta = 1} = -2\lambda\,\Omega(\hat\theta).$$
Unpenalized, maximum likelihood leaves $\tau^*_{\text{train}} = 1$; with an L2 penalty the model is underconfident on
its own training data. The log-linear bigram ($s_\beta(W) = \beta W$, $\Omega = \tfrac12\|W\|^2$) shows it to six
decimals; Topic 14's decoder (parameter-free final LayerNorm, tied unembedding at a fixed $1/\sqrt d$) has no such
direction, and its training-data $\tau^*$ is not 1.

In [ ]:
lc.test_scale_closed_stationarity()
lc.test_decoder_has_no_temperature_direction()
for lam in sorted(lc.LAMBDAS):
    k = f"bigram_{lam:g}"
    print(f"lambda = {lam:g}: dL/dbeta = {S[k + '_dbeta']:.6f}, -lambda||W||^2 = {S[k + '_minus_lam_norm']:.6f}, "
          f"tau* train / continuation / held = {S[k + '_train_tau_star']:.4f} / {S[k + '_continuation_tau_star']:.4f} / {S[k + '_held_tau_star']:.4f}")
print("decoder, training sentences: tau* =", round(S["decoder_train_tau_star"], 6))

## 6. Calibrated sets

**Proposition 3.** If $f$ is canonically calibrated, any set $S(F)$ chosen from the forecast covers $Y$ with probability
equal to its expected kept mass, so a nucleus of mass $p$ covers at least $p$. Coverage that differs from the kept mass
certifies miscalibration.

**Corollary 2.** Topic 15's sampling temperature, applied to a calibrated model at $\tau \ne 1$, produces a
miscalibrated sampler: the reliability term is $\mathbb E\,\mathrm{KL}(F \,\|\, F_\tau) \gt 0$. The same dial is
set for opposite reasons.

In [ ]:
lc.test_coverage_equals_kept_mass_when_calibrated()
lc.test_sampling_temperature_decalibrates()
lc.test_fast_nucleus_matches_topic15()
for e in lc.ENGINES:
    for t in lc.TEST_SETS:
        print(f"{e:8} {t:13} nucleus(0.9): coverage {S[f'{e}_{t}_cover90']:.4f} vs kept mass {S[f'{e}_{t}_kept90']:.4f}"
              f"   (at tau* {S[f'{e}_{t}_cover90_tau_star']:.4f})")

## 7. Finance case study

**(a) From synthetic MD&A to real filings.** In distribution, both engines sit close to the diagonal. On the public
EDGAR excerpts (most of their words unknown to both engines) both are badly overconfident; the NLL-optimal temperature
repairs the log loss by flattening every row, overshooting top-label calibration, and the nucleus still covers well
under half of the text.

**(b) Facts by how often they were seen.** Topic 15's order-5 Kneser–Ney fact model, restricted to a relation's
answers: facts seen twice or more are always right but under-confident; facts never seen are over-confident. Their
optimal temperatures sit at opposite ends of the grid, so one pooled temperature barely helps while one per regime
recovers half a nat.

In [ ]:
lc.test_in_distribution_nearly_calibrated()
lc.test_shift_can_go_either_way()
lc.test_edgar_overconfidence()
lc.test_fact_regimes_pull_apart()
for e in lc.ENGINES:
    k = f"{e}_edgar"
    print(f"{e:8} EDGAR: conf {S[k + '_conf']:.4f} vs acc {S[k + '_acc']:.4f}; ECE15 {S[k + '_ece15']:.4f} "
          f"(twin {S[k + '_twin15_mean']:.4f}); tau* {S[k + '_tau_star']:.4f}; cross-fitted gain {S[k + '_gain']:.4f} nats")
for g in lc.REGIMES + ("all",):
    print(f"facts {g:7} n = {S[f'fact_{g}_n']:3d}: conf {S[f'fact_{g}_conf']:.4f}, acc {S[f'fact_{g}_acc']:.4f}, "
          f"NLL {S[f'fact_{g}_nll']:.4f}, tau* {S[f'fact_{g}_tau_star']:.4f}")
print({k: round(v, 6) for k, v in lc.count_aware_nll().items()})

## 8. The lab's payload

`viz_payload()` is the single source of every number the D3 lab draws; its JSON mirror is pinned by
`test_viz_mirror_matches_payload`.

In [ ]:
lc.test_viz_payload()
lc.test_viz_mirror_matches_payload()
p = lc.viz_payload()
print(p["taus"], p["bins"], sorted(p["facts"]))

## Where this sits

Everything here is calibration of one next-token distribution at a time. Calibration of whole generations is a
different property: Braverman et al. (2020) show that the entropy rate of a language model's samples drifts upward
with length, and correct it with a one-parameter fit of its own. For large models the question moves to verbalized or
multiple-choice confidence (Kadavath et al. 2022), and post-training can undo a pretrained model's calibration
(OpenAI 2023, Fig. 8). What to *do* with a calibrated confidence (abstain, defer, or answer) is formalRAG's
selective-generation topic; a coverage guarantee that does not need calibration at all is conformal prediction
(formalML).